<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week2_Day2_Graph_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Day 2: State, schemas and a two-node graph

No LLM yet. This notebook shows:

1. The `Claim` and `AbstractExtraction` schemas (`src/pubmed_fetch/schemas.py`) accepting valid input and rejecting invalid input.
2. A LangGraph workflow (`src/pubmed_fetch/graph_basics.py`): `START -> load_abstract -> check_quotes -> END`.
3. The quote check flagging claims whose supporting quote is not in the abstract.

The claims below are written by hand. From Day 3 an LLM writes them, and this same check runs on its output.

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_USERNAME = "Guliko24"
REPO_NAME = "PubMed_Fetch"
BRANCH = "main"  # set to a branch name to test it before merging

repo_dir = Path("/content") / REPO_NAME
repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo_dir), *args], check=True, capture_output=True, text=True
    ).stdout.strip()

if not repo_dir.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(repo_dir)], check=True)
else:
    # /content survives "Restart session", so an older clone may be on another branch.
    git("fetch", "origin", BRANCH)
    git("checkout", BRANCH)
    git("pull", "--ff-only", "origin", BRANCH)

print(f"Clone at {repo_dir} is on branch {git('branch', '--show-current')}, commit {git('rev-parse', '--short', 'HEAD')}")

%pip -q install -r {repo_dir}/requirements.txt

# Until pyproject.toml arrives in Week 5, put src/ on the import path by hand.
for path in (repo_dir, repo_dir / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import search_utils
from pubmed_fetch import graph_basics, schemas
print("Package imported from:", schemas.__file__)

## 1. Load one abstract

In [ ]:
documents = search_utils.load_pubmed_data(repo_dir / "pubmed_abstracts.json")

PMID = "34582785"
abstract = next(doc["abstract"] for doc in documents if doc["pmid"] == PMID)
print(f"{len(documents)} documents loaded. PMID {PMID}:\n")
print(abstract)

## 2. The schemas

- `Claim` holds only what the LLM reads in the text. `claim_text` and `supporting_quote` are required and cannot be empty. `cell_type` and `disease` default to `None` and `marker_genes` to an empty list, which means *not stated*, never a guess.
- `AbstractExtraction` adds the `pmid`. The pipeline supplies it, not the LLM.
- Both forbid unknown fields and strip surrounding whitespace.

The JSON schema below is what Day 3 sends to the LLM, including each field's description and `additionalProperties: false`.

In [ ]:
import json

print(json.dumps(schemas.Claim.model_json_schema(), indent=2))

In [ ]:
from pydantic import ValidationError

invalid_inputs = {
    "missing claim_text and supporting_quote": {"cell_type": "MSNs"},
    "empty supporting_quote": {"claim_text": "MSNs exist.", "supporting_quote": "   "},
    "unknown field 'confidence'": {
        "claim_text": "MSNs exist.",
        "supporting_quote": "MSNs",
        "confidence": 0.9,
    },
}

# Every one of these must be refused.
for label, data in invalid_inputs.items():
    try:
        schemas.Claim(**data)
        print(f"ACCEPTED (unexpected): {label}")
    except ValidationError as error:
        print(f"Rejected: {label}")
        for problem in error.errors():
            print(f"    {problem['loc']}: {problem['msg']}")

## 3. Hand-written claims

| # | What it tests | Expected quote check |
|---|---|---|
| 1 | Exact quote, two markers (D1, D2) in one claim | found |
| 2 | True finding, but the quote is paraphrased | not found |
| 3 | Invented finding, no markers | not found |

In [ ]:
msn_quote = (
    "We identified discrete subpopulations of D1- and D2-expressing MSNs "
    "in the nucleus accumbens"
)

claims = [
    schemas.Claim(
        cell_type="MSNs",
        marker_genes=["D1", "D2"],
        claim_text="Discrete subpopulations of D1- and D2-expressing MSNs exist in the nucleus accumbens.",
        supporting_quote=msn_quote,
    ),
    schemas.Claim(
        cell_type="MSNs",
        marker_genes=["D1"],
        claim_text="D1-expressing MSN subpopulations are found in the nucleus accumbens.",
        supporting_quote="D1 MSN subpopulations were found in the nucleus accumbens",
    ),
    schemas.Claim(
        cell_type="microglia",
        disease="Alzheimer's disease",
        claim_text="Microglia are depleted in Alzheimer's disease.",
        supporting_quote="Microglia were depleted in Alzheimer's disease",
    ),
]

extraction = schemas.AbstractExtraction(pmid=PMID, claims=claims)
print(f"PMID {extraction.pmid}: {len(extraction.claims)} claims passed schema validation.")

## 4. Build and run the graph

In [ ]:
graph = graph_basics.build_graph(documents)

# The graph's structure as Mermaid text (paste into mermaid.live to draw it).
print(graph.get_graph().draw_mermaid())

In [ ]:
result = graph.invoke({"pmid": extraction.pmid, "claims": extraction.claims})

print("State keys after the run:", sorted(result))
print()
print(f"{'#':<3}{'markers':<10}{'cell type':<12}{'quote found':<13}claim")
for number, row in enumerate(result["checked"], start=1):
    claim = row["claim"]
    markers = ", ".join(claim.marker_genes) or "-"
    print(
        f"{number:<3}{markers:<10}{str(claim.cell_type):<12}"
        f"{str(row['quote_found']):<13}{claim.claim_text}"
    )

## What to check

- Section 2: all three invalid inputs are rejected; none print `ACCEPTED`.
- Claim 1 shows `True`; claims 2 and 3 show `False`, matching the table in section 3.
- Claim 2 is a true finding, yet it fails: the check is literal, so a paraphrased quote counts as unsupported. That is deliberate. The quote must be evidence copied from the abstract, not a summary.
- The final state contains `pmid` and `claims` (your input), `abstract` (added by `load_abstract`) and `checked` (added by `check_quotes`). Each node returns only the keys it changes.